# 网络优化与正则化

本节系统地把 PyTorch 训练时常用的几件工具串一遍：

1. **优化器** —— `torch.optim.{SGD, SGD+momentum, RMSprop, Adam}` 在一个二维损失面上的轨迹对比
2. **mini-batch 大小** 对收敛速度的影响
3. **参数初始化** —— Xavier / Kaiming，对深网络激活方差的稳定作用
4. **BatchNorm** —— 训练曲线对比
5. **Dropout** —— 作为正则化的效果
6. **学习率调度** —— `torch.optim.lr_scheduler`

为了让 notebook 跑得快，每个对比实验用的网络都不大、数据也少。结论方向在更大的网络上更明显。

## 1. 不同优化器在二维 loss 面上的轨迹

经典 Beale 函数：$f(x, y) = (1.5 - x + xy)^2 + (2.25 - x + xy^2)^2 + (2.625 - x + xy^3)^2$。
全局最小在 $(3, 0.5)$。从同一起点出发，看不同优化器走到哪里、走得多远。

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

def beale(x, y):
    return ((1.5 - x + x*y)**2
            + (2.25 - x + x*y**2)**2
            + (2.625 - x + x*y**3)**2)

def run_optim(opt_factory, n_steps=200):
    xy = torch.tensor([1.0, 1.5], requires_grad=True)
    opt = opt_factory([xy])
    traj = [xy.detach().clone().tolist()]
    for _ in range(n_steps):
        opt.zero_grad()
        beale(xy[0], xy[1]).backward()
        opt.step()
        traj.append(xy.detach().clone().tolist())
    return torch.tensor(traj)

optimizers = {
    'SGD lr=1e-2':         lambda p: optim.SGD(p, lr=1e-2),
    'SGD+momentum=0.9':    lambda p: optim.SGD(p, lr=1e-2, momentum=0.9),
    'RMSprop lr=1e-2':     lambda p: optim.RMSprop(p, lr=1e-2),
    'Adam lr=5e-2':        lambda p: optim.Adam(p, lr=5e-2),
}
trajs = {name: run_optim(fn) for name, fn in optimizers.items()}

# 绘制 loss 等高线 + 各优化器轨迹
xx, yy = torch.meshgrid(torch.linspace(-1, 4, 200), torch.linspace(-1, 3, 200), indexing='xy')
zz = beale(xx, yy)

plt.figure(figsize=(9, 6))
plt.contourf(xx, yy, torch.log10(zz + 1), levels=20, alpha=0.7)
plt.plot(3, 0.5, 'k*', ms=14, label='global min')
for name, t in trajs.items():
    plt.plot(t[:, 0], t[:, 1], '-o', ms=2, lw=1, label=name)
plt.legend(loc='upper right', fontsize=8); plt.xlabel('x'); plt.ylabel('y')
plt.title('optimizer trajectories on log10(Beale + 1)'); plt.show()

print('--- final f(x, y) ---')
for name, t in trajs.items():
    fx = beale(t[-1, 0], t[-1, 1]).item()
    print(f'  {name:25s}  final={t[-1].tolist()}  f={fx:.4f}')

**观察**：

- **SGD+momentum**、**RMSprop**、**Adam** 都成功逼近全局最小附近。Adam 需要比 SGD 大一个量级的 lr——这与 Adam 内部对梯度做 RMS 归一化的特性吻合。
- **纯 SGD** 步长太小、进度最慢。
- 这只是一个二维示例，结论方向：自适应优化器在病态曲率（如 Beale 函数细长的山谷）上有优势；momentum 在大多数 loss 面上有用。

### 教材版：带权 Sphere 函数上的逐个优化器轨迹

上面用 Beale 函数把几个优化器画在同一张图上。这里换成教材《神经网络与深度学习：案例与实践》第 7 章“2D 可视化实验”的写法：被优化函数取带权 Sphere 函数 $f(x)=w^{\top}x^2$（取 $w=[0.2,\,2]$，让两个维度的曲率不同），每个优化器单独画一张“等值线 + 更新轨迹”图。配色用 `cmap='rainbow'`：颜色越偏蓝表示函数值越小（最小的 $0\sim 8$ 区间为深蓝），越偏红表示函数值越大；蓝色折线是参数更新轨迹，红色五角星标出最优点 $x^{*}=0$。

In [ ]:
# 教材“2D 可视化实验”所需的被优化函数、训练循环与可视化工具。
# 优化器基类 Op / Optimizer / SimpleBatchGD 直接复用 nndl 包。
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
import torch
import numpy as np
import matplotlib.pyplot as plt
from nndl import Op, Optimizer, SimpleBatchGD


class OptimizedFunction(Op):
    """被优化函数：带权 Sphere  f(x)=wᵀx²，对 x 的偏导为 2·w⊙x。"""

    def __init__(self, w):
        super().__init__()
        self.w = w
        self.params = {'x': 0}
        self.grads = {'x': 0}

    def forward(self, x):
        self.params['x'] = x
        return torch.matmul(self.w, torch.square(self.params['x']))

    def backward(self):
        self.grads['x'] = 2 * torch.mul(self.w, self.params['x'])


def train_f(model, optimizer, x_init, epoch):
    """最简训练循环，记录每一步更新前的参数位置。"""
    x = x_init
    all_x, losses = [], []
    for i in range(epoch):
        all_x.append(x.clone())          # clone 一份，避免后续原地更新影响已记录的轨迹点
        loss = model(x)
        losses.append(loss.item())
        model.backward()
        optimizer.step()
        x = model.params['x']
    return torch.stack(all_x), losses


class Visualization:
    def __init__(self):
        # 只画参数 x1、x2 在区间 [-5, 5] 的部分
        x1 = np.arange(-5, 5, 0.1)
        x2 = np.arange(-5, 5, 0.1)
        x1, x2 = np.meshgrid(x1, x2)
        self.init_x = torch.tensor(np.array([x1, x2]), dtype=torch.float32)

    def plot_2d(self, model, x):
        """把函数等值线与优化轨迹画在同一张图上。"""
        fig, ax = plt.subplots(figsize=(10, 6))
        x1, x2 = self.init_x[0].numpy(), self.init_x[1].numpy()
        # 网格上逐点的函数值，等价于对每个 (x1, x2) 求 f(x)=wᵀx²
        z = (model.w[0] * self.init_x[0] ** 2 + model.w[1] * self.init_x[1] ** 2).numpy()
        cp = ax.contourf(x1, x2, z, cmap='rainbow')
        ax.contour(x1, x2, z, colors='black')
        fig.colorbar(cp)
        traj = x.detach().numpy()
        ax.plot(traj[:, 0], traj[:, 1], '-o', color='b')   # 蓝色折线：参数更新轨迹
        ax.plot(0, 0, 'r*', markersize=18)                  # 红色五角星：最优点
        ax.set_xlabel('$x1$')
        ax.set_ylabel('$x2$')
        ax.set_xlim((-2, 5))
        ax.set_ylim((-2, 5))


def train_and_plot_f(model, optimizer, epoch):
    """训练模型并可视化参数更新轨迹。"""
    x_init = torch.tensor([3, 4], dtype=torch.float32)
    print(f'x1 initiate: {x_init[0].numpy()}, x2 initiate: {x_init[1].numpy()}')
    x, losses = train_f(model, optimizer, x_init, epoch)
    Visualization().plot_2d(model, x)

In [ ]:
# 梯度下降法：复用 nndl 的 SimpleBatchGD，对应图 fig:opti-vis-para
torch.manual_seed(0)
w = torch.tensor([0.2, 2])
model = OptimizedFunction(w)
opt = SimpleBatchGD(init_lr=0.2, model=model)
train_and_plot_f(model, opt, epoch=20)

In [ ]:
class Adagrad(Optimizer):
    def __init__(self, init_lr, model, epsilon):
        super().__init__(init_lr=init_lr, model=model)
        self.G = {key: 0 for key in self.model.params.keys()}
        self.epsilon = epsilon

    def adagrad(self, x, gradient_x, G, init_lr):
        # G 为参数梯度平方的累积值（只增不减）
        G = G + gradient_x ** 2
        x = x - init_lr / torch.sqrt(G + self.epsilon) * gradient_x
        return x, G

    def step(self):
        for key in self.model.params.keys():
            self.model.params[key], self.G[key] = self.adagrad(
                self.model.params[key], self.model.grads[key], self.G[key], self.init_lr)


# AdaGrad，对应图 fig:opti-vis-para2
torch.manual_seed(0)
w = torch.tensor([0.2, 2])
model = OptimizedFunction(w)
opt = Adagrad(init_lr=0.5, model=model, epsilon=1e-7)
train_and_plot_f(model, opt, epoch=50)

In [ ]:
class RMSprop(Optimizer):
    def __init__(self, init_lr, model, beta, epsilon):
        super().__init__(init_lr=init_lr, model=model)
        self.G = {key: 0 for key in self.model.params.keys()}
        self.beta = beta
        self.epsilon = epsilon

    def rmsprop(self, x, gradient_x, G, init_lr):
        # G 为梯度平方的指数加权移动平均（可升可降）
        G = self.beta * G + (1 - self.beta) * gradient_x ** 2
        x = x - init_lr / torch.sqrt(G + self.epsilon) * gradient_x
        return x, G

    def step(self):
        for key in self.model.params.keys():
            self.model.params[key], self.G[key] = self.rmsprop(
                self.model.params[key], self.model.grads[key], self.G[key], self.init_lr)


# RMSprop，对应图 fig:opti-vis-para3
torch.manual_seed(0)
w = torch.tensor([0.2, 2])
model = OptimizedFunction(w)
opt = RMSprop(init_lr=0.1, model=model, beta=0.9, epsilon=1e-7)
train_and_plot_f(model, opt, epoch=50)

In [ ]:
class Momentum(Optimizer):
    def __init__(self, init_lr, model, rho):
        super().__init__(init_lr=init_lr, model=model)
        self.delta_x = {key: 0 for key in self.model.params.keys()}
        self.rho = rho

    def momentum(self, x, gradient_x, delta_x, init_lr):
        # delta_x 为负梯度的指数加权移动平均（动量）
        delta_x = self.rho * delta_x - init_lr * gradient_x
        x = x + delta_x
        return x, delta_x

    def step(self):
        for key in self.model.params.keys():
            self.model.params[key], self.delta_x[key] = self.momentum(
                self.model.params[key], self.model.grads[key], self.delta_x[key], self.init_lr)


# 动量法，对应图 fig:opti-vis-para4
torch.manual_seed(0)
w = torch.tensor([0.2, 2])
model = OptimizedFunction(w)
opt = Momentum(init_lr=0.01, model=model, rho=0.9)
train_and_plot_f(model, opt, epoch=50)

In [ ]:
class Adam(Optimizer):
    def __init__(self, init_lr, model, beta1, beta2, epsilon):
        super().__init__(init_lr=init_lr, model=model)
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.M = {key: 0 for key in self.model.params.keys()}
        self.G = {key: 0 for key in self.model.params.keys()}
        self.t = 1

    def adam(self, x, gradient_x, G, M, t, init_lr):
        M = self.beta1 * M + (1 - self.beta1) * gradient_x          # 一阶矩（动量）
        G = self.beta2 * G + (1 - self.beta2) * gradient_x ** 2     # 二阶矩（自适应步长）
        M_hat = M / (1 - self.beta1 ** t)                           # 偏差修正
        G_hat = G / (1 - self.beta2 ** t)
        t += 1
        x = x - init_lr / torch.sqrt(G_hat + self.epsilon) * M_hat
        return x, G, M, t

    def step(self):
        for key in self.model.params.keys():
            self.model.params[key], self.G[key], self.M[key], self.t = self.adam(
                self.model.params[key], self.model.grads[key], self.G[key], self.M[key],
                self.t, self.init_lr)


# Adam，对应图 fig:opti-vis-para5
torch.manual_seed(0)
w = torch.tensor([0.2, 2])
model = OptimizedFunction(w)
opt = Adam(init_lr=0.2, model=model, beta1=0.9, beta2=0.99, epsilon=1e-7)
train_and_plot_f(model, opt, epoch=20)

### 教材版：不同优化器的3D可视化对比（鞍点）

把上面的二维实验换到三维场景。被优化函数取 $f(x)=x_0^2+x_1^2+x_1^3+x_0\cdot x_1$，它在 $(0,0)$ 附近存在**鞍点**——既非极大值也非极小值的临界点。关注点是：不同优化器能否绕开鞍点继续向真正的最小值靠拢，对应教材图 opti-f-3d 与 7-15。下面复用本节已经定义好的 `SimpleBatchGD`、`Adagrad`、`RMSprop`、`Momentum`、`Adam` 五个优化器。

<!-- MIGRATED-FROM-BOOK: 3d-saddle-and-decision-boundary -->

In [ ]:
# 三维被优化函数：f(x)=x0²+x1²+x1³+x0·x1，(0,0) 附近是鞍点
# 继承 nndl 的 Op；backward 手动给出对 x0、x1 的偏导。
class OptimizedFunction3D(Op):
    def __init__(self):
        super().__init__()
        self.params = {'x': 0}
        self.grads = {'x': 0}

    def forward(self, x):
        self.params['x'] = x
        return x[0] ** 2 + x[1] ** 2 + x[1] ** 3 + x[0] * x[1]

    def backward(self):
        x = self.params['x']
        gradient1 = 2 * x[0] + x[1]
        gradient2 = 2 * x[1] + 3 * x[1] ** 2 + x[0]
        # 两个分量都是 0 维标量，用 torch.stack 拼成 1 维梯度向量
        # （教材里写的 torch.cat 只接受 1 维张量，这里改成 stack）
        self.grads['x'] = torch.stack([gradient1, gradient2])

In [ ]:
# 5 个模型分别配 5 个优化器，在同一函数上各跑 150 步，记录参数轨迹
model1 = OptimizedFunction3D()
opt_gd = SimpleBatchGD(init_lr=0.01, model=model1)
model2 = OptimizedFunction3D()
opt_adagrad = Adagrad(init_lr=0.5, model=model2, epsilon=1e-7)
model3 = OptimizedFunction3D()
opt_rmsprop = RMSprop(init_lr=0.1, model=model3, beta=0.9, epsilon=1e-7)
model4 = OptimizedFunction3D()
opt_momentum = Momentum(init_lr=0.01, model=model4, rho=0.9)
model5 = OptimizedFunction3D()
opt_adam = Adam(init_lr=0.1, model=model5, beta1=0.9, beta2=0.99, epsilon=1e-7)

models = [model1, model2, model3, model4, model5]
opts = [opt_gd, opt_adagrad, opt_rmsprop, opt_momentum, opt_adam]

x_all_opts, z_all_opts = [], []
x_init = torch.tensor([2, 3], dtype=torch.float32)
for model, opt in zip(models, opts):
    x_one_opt, z_one_opt = train_f(model, opt, x_init, 150)   # 复用上面定义的 train_f
    x_all_opts.append(x_one_opt.numpy())
    z_all_opts.append(np.squeeze(z_one_opt))
print('每个优化器的轨迹形状：', [a.shape for a in x_all_opts])

In [ ]:
# 先把被优化函数的 3D 曲面绘出来，作为动画背景，对应图 opti-f-3d
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (注册 3d 投影)

# 用 numpy.meshgrid 在 [-3, 3]×[-3, 3] 上以 0.1 为间隔生成网格
x1 = np.arange(-3, 3, 0.1)
x2 = np.arange(-3, 3, 0.1)
x1, x2 = np.meshgrid(x1, x2)
init_x = torch.tensor(np.array([x1, x2]), dtype=torch.float32)
model = OptimizedFunction3D()

fig = plt.figure()
ax = plt.axes(projection='3d')
# model(init_x) 返回 torch 张量，转成 numpy 再交给 plot_surface
ax.plot_surface(init_x[0].numpy(), init_x[1].numpy(), model(init_x).numpy(), cmap='rainbow')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_zlabel('f(x1,x2)')
plt.show()

In [ ]:
# 封装 Visualization3D：把 5 条参数轨迹叠成一段 3D 动画，对应图 7-15
from matplotlib import animation
from itertools import zip_longest


class Visualization3D(animation.FuncAnimation):
    """绘制动态图像，可视化参数更新轨迹。"""

    def __init__(self, *xy_values, z_values, labels=[], colors=[], fig, ax,
                 interval=60, blit=True, **kwargs):
        """
        输入：
            xy_values：各优化器在 x、y 维度上的轨迹
            z_values：各优化器对应的 z（函数值）轨迹
            labels：每条轨迹的标签
            colors：每条轨迹的颜色
            interval：帧之间的延迟（毫秒）
            blit：是否优化绘图
        """
        self.fig = fig
        self.ax = ax
        self.xy_values = xy_values
        self.z_values = z_values
        frames = max(xy_value.shape[0] for xy_value in xy_values)
        self.lines = [ax.plot([], [], [], label=label, color=color, lw=2)[0]
                      for _, label, color in zip_longest(xy_values, labels, colors)]
        super().__init__(fig, self.animate, init_func=self.init_animation,
                         frames=frames, interval=interval, blit=blit, **kwargs)

    def init_animation(self):
        # 数值初始化
        for line in self.lines:
            line.set_data([], [])
            line.set_3d_properties([])
        return self.lines

    def animate(self, i):
        # 将 x、y、z 三个数据传入，绘制三维图像
        for line, xy_value, z_value in zip(self.lines, self.xy_values, self.z_values):
            line.set_data(xy_value[:i, 0], xy_value[:i, 1])
            line.set_3d_properties(z_value[:i])
        return self.lines


# 构造动画对象并手动触发一次 init / 单帧绘制做自检（无需真渲染视频或保存 mp4）。
# 在 Notebook 里若要看动画，可执行：from IPython.display import HTML; HTML(anim.to_html5_video())
labels = ['SGD', 'AdaGrad', 'RMSprop', 'Momentum', 'Adam']
colors = ['r', 'b', 'y', 'g', 'c']
anim = Visualization3D(*x_all_opts, z_values=z_all_opts, labels=labels, colors=colors, fig=fig, ax=ax)
anim.init_animation()
anim.animate(10)
ax.legend(loc='upper left')
print('Visualization3D 构造并完成 init_animation/animate 自检')

从动画可以看到：在这个特定函数上，Momentum 利用历史动量绕开了鞍点，而其余几种优化器在150步内仍停留在鞍点附近。这只能说明 Momentum 在该函数上有优势，并不代表它是“最佳优化器”——具体任务里采用哪种优化器，仍要结合数据规模、损失曲面形态和工程预算综合权衡。

### 教材版：正则化的决策边界可视化（show_class_boundary）

接下来把教材“网络正则化”一节的决策边界绘图 `show_class_boundary` 迁移过来（对应图 opti-regularization1–4）。为了让 `show_class_boundary(model, X, y, *args)` 这种 `Op` 风格的位置调用能直接复用，这里也把教材里基于自定义 `Op` 的三层感知器 `MLP_3L`、`ReLU` 算子、带对率的二分类交叉熵 `BinaryCrossEntropyWithLogits` 一并 port 过来，在合成 `make_moons` 数据集上做一次轻量训练后画出决策面。

In [ ]:
# 决策边界演示所需的算子：ReLU、单层线性、三层 MLP、带对率的交叉熵
from nndl.data import make_moons


class ReLU(Op):
    """ReLU 算子：前向逐元素取正部，反向按“前向是否>0”的掩码透传上游梯度。"""

    def __init__(self):
        self.inputs = None

    def forward(self, inputs):
        self.inputs = inputs
        return inputs.clamp(min=0)

    def backward(self, outputs_grads):
        return outputs_grads * (self.inputs > 0).to(outputs_grads.dtype)


class LinearLayer(Op):
    """单层线性算子 z = X·W + b，手写前向/反向（梯度写进 self.grads）。"""

    def __init__(self, input_size, output_size, name='fc'):
        super().__init__()
        self.name = name
        self.params = {'W': torch.randn(input_size, output_size),
                       'b': torch.zeros(output_size)}
        self.grads = {}
        self.inputs = None

    def forward(self, inputs):
        self.inputs = inputs
        return torch.matmul(inputs, self.params['W']) + self.params['b']

    def backward(self, grads):
        self.grads['W'] = torch.matmul(self.inputs.t(), grads)
        self.grads['b'] = torch.sum(grads, dim=0)
        return torch.matmul(grads, self.params['W'].t())


class MLP_3L(Op):
    """三层 MLP：两层 Linear+ReLU，末层 Linear 直接输出对率（Logits）。"""

    def __init__(self, layers_size):
        self.fc1 = LinearLayer(layers_size[0], layers_size[1], name='fc1')
        self.act_fn1 = ReLU()
        self.fc2 = LinearLayer(layers_size[1], layers_size[2], name='fc2')
        self.act_fn2 = ReLU()
        self.fc3 = LinearLayer(layers_size[2], layers_size[3], name='fc3')
        self.layers = [self.fc1, self.act_fn1, self.fc2, self.act_fn2, self.fc3]

    def forward(self, X):
        z1 = self.fc1(X)
        a1 = self.act_fn1(z1)
        z2 = self.fc2(a1)
        a2 = self.act_fn2(z2)
        z3 = self.fc3(a2)
        return z3

    def backward(self, loss_grad_z3):
        g = self.fc3.backward(loss_grad_z3)
        g = self.act_fn2.backward(g)
        g = self.fc2.backward(g)
        g = self.act_fn1.backward(g)
        self.fc1.backward(g)


class BinaryCrossEntropyWithLogits(Op):
    """对率输入的二分类交叉熵：内部先 sigmoid 再算交叉熵，反向回传到 model。"""

    def __init__(self, model):
        self.model = model

    def forward(self, logits, labels):
        self.predicts = torch.sigmoid(logits)
        self.labels = labels
        self.data_size = self.predicts.shape[0]
        eps = 1e-9  # 避免 log(0)
        loss = -1. / self.data_size * (
            torch.matmul(self.labels.t(), torch.log(self.predicts + eps)) +
            torch.matmul((1 - self.labels.t()), torch.log(1 - self.predicts + eps)))
        return torch.squeeze(loss, dim=1)

    def backward(self):
        inputs_grads = 1. / self.data_size * (self.predicts - self.labels)
        self.model.backward(inputs_grads)


class MultiLayerGD(Optimizer):
    """遍历模型每一层、逐参数做小批量梯度下降（MLP_3L 的参数分散在各层里）。"""

    def step(self):
        for layer in self.model.layers:
            if hasattr(layer, 'params') and isinstance(layer.params, dict):
                for key in layer.params.keys():
                    layer.params[key] = layer.params[key] - self.init_lr * layer.grads[key]


def accuracy_logits(logits, labels):
    """直接从对率算准确率：二分类按是否大于0判定，多分类按 argmax。"""
    if logits.shape[1] == 1:
        preds = (logits > 0).to(torch.float32)
    else:
        preds = torch.argmax(logits, dim=1).to(torch.int32)
    return torch.mean(torch.eq(preds, labels).to(torch.float32))

In [ ]:
import matplotlib.pyplot as plt


def show_class_boundary(model, X_train, y_train, *args):
    """在网格上密集采样并预测类别，画出决策区域 + 训练样本散点。"""
    # 均匀生成 40 000 个数据点；torch 新版 meshgrid 需显式 indexing='ij'
    x1, x2 = torch.meshgrid(torch.linspace(-2, 3, 200),
                            torch.linspace(-3, 3, 200), indexing='ij')
    x = torch.stack([torch.flatten(x1), torch.flatten(x2)], dim=1)
    # 预测对应类别
    y = model(x, *args)
    y = (y > 0).to(torch.int32).squeeze()
    # 绘制类别区域
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.scatter(x[:, 0].numpy(), x[:, 1].numpy(), c=y.tolist(), cmap=plt.cm.Spectral)
    plt.scatter(X_train[:, 0].numpy(), X_train[:, 1].numpy(), marker='*',
                c=y_train.squeeze().tolist())

In [ ]:
# 在合成 make_moons 上做一次轻量训练，再画决策边界（对应图 opti-regularization）
torch.manual_seed(0)
data_X, data_y = make_moons(n_samples=300, shuffle=True, noise=0.5)
X_train, y_train = data_X[:200], data_y[:200].reshape([-1, 1])

layers_size = [X_train.shape[1], 20, 3, 1]
model = MLP_3L(layers_size)
opt = MultiLayerGD(init_lr=0.2, model=model)
loss_fn = BinaryCrossEntropyWithLogits(model)

# tiny 训练：演示用只跑几十步（教材正式实验是 50000 步）
for i in range(50):
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    loss_fn.backward()
    opt.step()

print('训练集准确率：', accuracy_logits(model(X_train), y_train).numpy())

plt.figure()
show_class_boundary(model, X_train, y_train)
plt.show()

## 2. mini-batch 大小对收敛速度的影响

用 LeNet 在 MNIST 子集上，固定优化器、固定 lr，只改 batch_size。

In [ ]:
import os
from torch.utils.data import Subset
from torchvision import datasets, transforms

CACHE = os.path.expanduser('~/.cache/torch_data')
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_full = datasets.MNIST(CACHE, train=True, download=True, transform=tfm)
test_full  = datasets.MNIST(CACHE, train=False, download=True, transform=tfm)

torch.manual_seed(0)
tr_idx = torch.randperm(len(train_full))[:5000].tolist()
te_idx = torch.randperm(len(test_full))[:1000].tolist()
train_set = Subset(train_full, tr_idx)
test_set  = Subset(test_full,  te_idx)
test_loader = DataLoader(test_set, batch_size=256)

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))

from nndl import LeNet5

from nndl.runner import RunnerV3


In [ ]:
from nndl import accuracy

def train_with_bs(bs, epochs=3, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    loader = DataLoader(train_set, batch_size=bs, shuffle=True)
    model = LeNet5()
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    runner = RunnerV3(model, opt, nn.CrossEntropyLoss(),
                      metric_fn=accuracy, higher_is_better=True)
    runner.fit(loader, test_loader, num_epochs=epochs, log_every=None)
    return runner.history['dev_metric']

bs_accs = {bs: train_with_bs(bs) for bs in [16, 64, 256]}
for bs, accs in bs_accs.items():
    print(f'bs={bs:4d}  test_acc by epoch: ' + ', '.join(f'{a:.3f}' for a in accs))

小 batch 每个 epoch 走更多步（同样数据），所以收敛**早**——但 batch 太小时单步梯度方差大，最终精度不一定更好。Practical 选择常是 32-256。

## 3. 参数初始化：Xavier vs Kaiming

目标：信号通过深网络后，每层的方差应当大致保持稳定（既不爆炸也不消失）。

- **Xavier (`xavier_normal_`)**: 假设 tanh / sigmoid 这样的近线性激活，方差按 $2 / (n_\text{in} + n_\text{out})$ 缩放。
- **Kaiming (`kaiming_normal_`)**: 假设 ReLU 类激活（一半被砍掉），方差按 $2 / n_\text{in}$ 缩放。

下面构造一个 10 层的全连接网络，看不同初始化下每层输出的标准差。

In [ ]:
def build_deep_mlp(init_kind, n_layers=10, dim=512, activation='relu'):
    torch.manual_seed(0)
    layers = []
    for _ in range(n_layers):
        layers.append(nn.Linear(dim, dim))
        layers.append(nn.ReLU() if activation == 'relu' else nn.Tanh())
    model = nn.Sequential(*layers)
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if init_kind == 'naive':
                nn.init.normal_(m.weight, std=0.05)
            elif init_kind == 'xavier':
                nn.init.xavier_normal_(m.weight)
            elif init_kind == 'kaiming':
                nn.init.kaiming_normal_(m.weight, nonlinearity=activation)
            nn.init.zeros_(m.bias)
    return model

def collect_std(model):
    """Forward 一次，收集每个 Linear 层输出的标准差。"""
    x = torch.randn(64, 512)
    stds = []
    with torch.no_grad():
        for m in model:
            x = m(x)
            if isinstance(m, nn.Linear):
                stds.append(x.std().item())
    return stds

for kind in ['naive', 'xavier', 'kaiming']:
    stds = collect_std(build_deep_mlp(kind, activation='relu'))
    print(f'{kind:8s} (ReLU):  ' + ', '.join(f'{s:7.3f}' for s in stds))

**naive 高斯初始化** 让输出方差按 ~10x 速度衰减；**Xavier** 偏小（针对 tanh 设计）；**Kaiming** 在 ReLU 下能保持每层方差在同一量级。这就是深层 ReLU 网络默认用 Kaiming 的原因（PyTorch `nn.Linear` 的默认就是 Kaiming uniform）。

## 4. BatchNorm 的训练加速效果

BN 在 batch 维度上把每个特征通道**归一化**后再线性变换。它能让我们使用更大的学习率、加快收敛，并自带一些正则化效果。

In [ ]:
class MLPNoBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128), nn.ReLU(),
            nn.Linear(128,    64), nn.ReLU(),
            nn.Linear(64,     10),
        )
    def forward(self, x): return self.net(x)


class MLPWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128,    64), nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Linear(64,     10),
        )
    def forward(self, x): return self.net(x)


def train_mlp(model, epochs=3, lr=1e-1, bs=64):
    torch.manual_seed(0)
    loader = DataLoader(train_set, batch_size=bs, shuffle=True)
    opt = optim.SGD(model.parameters(), lr=lr)
    # 本节只关心 train_loss 曲线，不传 dev_loader——RunnerV3 会跳过 dev 评估。
    runner = RunnerV3(model, opt, nn.CrossEntropyLoss())
    runner.fit(loader, dev_loader=None, num_epochs=epochs, log_every=None)
    return runner.history['train_loss']


no_bn = train_mlp(MLPNoBN())
with_bn = train_mlp(MLPWithBN())
for ep, (a, b) in enumerate(zip(no_bn, with_bn), 1):
    print(f'epoch {ep}: no-BN train_loss={a:.4f}   with-BN={b:.4f}')

在大学习率（`lr=0.1`）下，无 BN 的网络收敛慢甚至发散；BN 让它稳稳下降。这是 BN 最常被引用的好处。

## 5. Dropout 的正则化效果

在一个会过拟合的小数据 / 大网络组合上对比。

In [ ]:
# 小数据：MNIST 1000 训练样本
tiny_train = Subset(train_full, torch.randperm(len(train_full))[:1000].tolist())
tiny_train_loader = DataLoader(tiny_train, batch_size=64, shuffle=True)


class BigMLP(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,   256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,    10),
        )
    def forward(self, x): return self.net(x)


def train_and_eval(model, epochs=15, lr=1e-2):
    torch.manual_seed(0)
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    runner = RunnerV3(model, opt, nn.CrossEntropyLoss(),
                      metric_fn=lambda out, y: (out.argmax(1) == y).float().mean().item(),
                      higher_is_better=True)
    train_acc, test_acc = [], []
    # 每 epoch 在 train 与 test 上各 evaluate 一次，绘制泛化 gap 曲线
    for _ in range(epochs):
        runner.fit(tiny_train_loader, test_loader, num_epochs=1, log_every=None)
        _, tr_a = runner.evaluate(tiny_train_loader)
        train_acc.append(tr_a)
        test_acc.append(runner.history['dev_metric'][-1])
    return train_acc, test_acc


tr_no, te_no = train_and_eval(BigMLP(dropout=0.0))
tr_dp, te_dp = train_and_eval(BigMLP(dropout=0.5))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(tr_no, '-o', label='train (no dropout)')
ax1.plot(te_no, '-s', label='test  (no dropout)')
ax1.set_title('No Dropout'); ax1.legend(); ax1.grid(alpha=0.3); ax1.set_xlabel('epoch')

ax2.plot(tr_dp, '-o', label='train (dropout=0.5)')
ax2.plot(te_dp, '-s', label='test  (dropout=0.5)')
ax2.set_title('Dropout 0.5'); ax2.legend(); ax2.grid(alpha=0.3); ax2.set_xlabel('epoch')
plt.tight_layout(); plt.show()

Dropout 让训练-测试准确率差距（generalization gap）变小，是经典正则化手段。注意 `model.eval()` 会自动关掉 dropout，所以 `runner.evaluate()` 不需要手动处理。

## 6. 学习率调度

`torch.optim.lr_scheduler` 提供了一系列调度器。常用：

- **`StepLR(gamma=0.1, step_size=30)`**：每 30 epoch 把 lr × 0.1
- **`CosineAnnealingLR(T_max=...)`**：余弦衰减（现代默认）
- **`ReduceLROnPlateau`**：dev 指标停止改进时降 lr

下面画三种调度器在 100 epoch 上的 lr 曲线。

In [ ]:
model = nn.Linear(2, 2)
epochs = 100
schedulers = {
    'StepLR(30, 0.5)':            lambda: optim.lr_scheduler.StepLR(optim.SGD(model.parameters(), lr=0.1), step_size=30, gamma=0.5),
    'CosineAnnealingLR(100)':     lambda: optim.lr_scheduler.CosineAnnealingLR(optim.SGD(model.parameters(), lr=0.1), T_max=epochs),
    'ExponentialLR(0.97)':        lambda: optim.lr_scheduler.ExponentialLR(optim.SGD(model.parameters(), lr=0.1), gamma=0.97),
}

plt.figure(figsize=(9, 4))
for name, factory in schedulers.items():
    sched = factory()
    opt_ref = sched.optimizer
    lrs = [opt_ref.param_groups[0]['lr']]
    for _ in range(epochs):
        opt_ref.step()                    # 实际训练中这一步是真正用 lr 的地方
        sched.step()
        lrs.append(opt_ref.param_groups[0]['lr'])
    plt.plot(lrs, label=name)
plt.xlabel('epoch'); plt.ylabel('learning rate'); plt.legend(); plt.grid(alpha=0.3); plt.show()

**典型使用模板**：

```python
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
for epoch in range(epochs):
    train_one_epoch(model, optimizer, ...)
    scheduler.step()                # 注意：scheduler.step() 一般 *per-epoch*
```

## 小结

- **优化器**：Adam 是无脑默认；SGD+Momentum 在 CV 大数据上微调常超过 Adam。
- **batch size**：32-256 是甜区。太小方差大，太大每 epoch 步数太少。
- **初始化**：用 `nn.init.kaiming_normal_/uniform_` 配 ReLU；`xavier_*` 配 tanh/sigmoid。PyTorch `nn.Linear/Conv2d` 默认就用 Kaiming uniform——大多数情况下不用手动设。
- **BatchNorm**：现代 CNN 标配；让大学习率成为可能。注意 `train()` / `eval()` 模式切换。
- **Dropout**：过拟合时祭出，0.2-0.5 之间挑。`eval()` 自动关掉。
- **lr_scheduler**：CosineAnnealingLR / OneCycleLR 是现代默认；`scheduler.step()` 通常每 epoch 调一次。
- **weight decay**：直接挂在优化器上 (`weight_decay=...`)，等价于 L2 正则。CV 常用 5e-4，Transformer 常用 0.1。